In [0]:
%sql
SELECT order_id, customer_id, order_purchase_timestamp, order_status
FROM read_files(
  '/Volumes/olist/source/raw_files/olist_orders_dataset.csv',
  format => 'csv',
  header => true
)
LIMIT 5

order_id,customer_id,order_purchase_timestamp,order_status
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02T10:56:33.000Z,delivered
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24T20:41:37.000Z,delivered
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08T08:38:49.000Z,delivered
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18T19:28:06.000Z,delivered
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13T21:18:39.000Z,delivered


In [0]:
%sql
SELECT COUNT(*) AS january_orders
FROM read_files(
  '/Volumes/olist/source/raw_files/olist_orders_dataset.csv',
  format => 'csv',
  header => true
)
WHERE order_purchase_timestamp >= TIMESTAMP '2018-01-01 00:00:00'
  AND order_purchase_timestamp <  TIMESTAMP '2018-02-01 00:00:00'

january_orders
7269


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS olist.bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS olist.bronze.orders
USING DELTA
AS
SELECT *, DATE '2018-01-01' AS purchase_month
FROM read_files(
  '/Volumes/olist/source/raw_files/olist_orders_dataset.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false,
  schemaEvolutionMode => 'none'
)
WHERE CAST(order_purchase_timestamp AS TIMESTAMP) >= TIMESTAMP '2018-01-01 00:00:00'
  AND CAST(order_purchase_timestamp AS TIMESTAMP) <  TIMESTAMP '2018-02-01 00:00:00'

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT purchase_month, COUNT(*) AS order_count
FROM olist.bronze.orders
GROUP BY purchase_month

purchase_month,order_count
2018-01-01,7269
2018-03-01,7211
2018-02-01,6728


In [0]:
%sql
CREATE TABLE IF NOT EXISTS olist.bronze.customers
USING DELTA
AS
SELECT *
FROM read_files(
  '/Volumes/olist/source/raw_files/olist_customers_dataset.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false,
  schemaEvolutionMode => 'none'
)

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS customer_rows,
       COUNT(DISTINCT customer_id) AS distinct_customer_ids
FROM olist.bronze.customers

customer_rows,distinct_customer_ids
99441,99441


In [0]:
%sql
SELECT
  COUNT(*) AS joined_rows,
  COUNT(DISTINCT o.order_id) AS distinct_orders,
  COUNT(*) - COUNT(c.customer_id) AS unmatched_orders
FROM olist.bronze.orders AS o
LEFT JOIN olist.bronze.customers AS c
  ON o.customer_id = c.customer_id

joined_rows,distinct_orders,unmatched_orders
7269,7269,0


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS olist.silver

In [0]:
%sql
CREATE OR REPLACE TABLE olist.silver.orders_enriched
USING DELTA
AS
WITH typed_orders AS (
  SELECT
    o.order_id,
    o.customer_id,
    c.customer_state,
    o.order_status,
    o.purchase_month,
    CAST(o.order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
    CAST(o.order_delivered_customer_date AS DATE) AS actual_delivery_date,
    CAST(o.order_estimated_delivery_date AS DATE) AS estimated_delivery_date
  FROM olist.bronze.orders AS o
  LEFT JOIN olist.bronze.customers AS c
    ON o.customer_id = c.customer_id
)
SELECT *,
  CASE WHEN order_status = 'delivered'
    THEN DATEDIFF(actual_delivery_date, estimated_delivery_date)
  END AS days_vs_estimate
FROM typed_orders

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
  COUNT(*) AS order_count,
  COUNT(days_vs_estimate) AS eligible_deliveries,
  COUNT(CASE WHEN days_vs_estimate > 0 THEN 1 END) AS late_deliveries
FROM olist.silver.orders_enriched

order_count,eligible_deliveries,late_deliveries
21208,20627,2657


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS olist.gold

In [0]:
%sql
CREATE OR REPLACE TABLE olist.gold.monthly_delivery_performance
USING DELTA
AS
SELECT
  purchase_month,
  customer_state,
  COUNT(*) AS order_count,
  COUNT(CASE WHEN order_status = 'delivered' THEN 1 END) AS delivered_orders,
  COUNT(days_vs_estimate) AS eligible_deliveries,
  COUNT(CASE WHEN days_vs_estimate > 0 THEN 1 END) AS late_deliveries,
  SUM(CASE WHEN days_vs_estimate > 0 THEN days_vs_estimate ELSE 0 END) AS total_days_late
FROM olist.silver.orders_enriched
GROUP BY purchase_month, customer_state

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
  purchase_month,
  SUM(order_count) AS order_count,
  SUM(eligible_deliveries) AS eligible_deliveries,
  SUM(late_deliveries) AS late_deliveries
FROM olist.gold.monthly_delivery_performance
GROUP BY purchase_month

purchase_month,order_count,eligible_deliveries,late_deliveries
2018-01-01,7269,7069,403


In [0]:
%sql
SELECT
  purchase_month,
  SUM(order_count) AS orders_placed,
  SUM(delivered_orders) AS orders_delivered
FROM olist.gold.monthly_delivery_performance
GROUP BY purchase_month
ORDER BY purchase_month

purchase_month,orders_placed,orders_delivered
2018-01-01,7269,7069


In [0]:
%sql
SELECT
  purchase_month,
  SUM(eligible_deliveries) AS eligible_deliveries,
  SUM(late_deliveries) AS late_deliveries,
  ROUND(100.0 * SUM(late_deliveries) / NULLIF(SUM(eligible_deliveries), 0), 2)
    AS late_delivery_pct,
  ROUND(1.0 * SUM(total_days_late) / NULLIF(SUM(late_deliveries), 0), 2)
    AS avg_days_late
FROM olist.gold.monthly_delivery_performance
GROUP BY purchase_month
ORDER BY purchase_month

purchase_month,eligible_deliveries,late_deliveries,late_delivery_pct,avg_days_late
2018-01-01,7069,403,5.70,11.88
2018-02-01,6555,926,14.13,12.58
2018-03-01,7003,1328,18.96,10.02
2018-04-01,6798,306,4.50,10.40


In [0]:
%sql
SELECT
  customer_state,
  eligible_deliveries,
  late_deliveries,
  ROUND(100.0 * late_deliveries / eligible_deliveries, 2) AS late_delivery_pct
FROM olist.gold.monthly_delivery_performance
WHERE purchase_month = DATE '2018-01-01'
  AND eligible_deliveries > 0
ORDER BY late_delivery_pct DESC, eligible_deliveries DESC, customer_state

customer_state,eligible_deliveries,late_deliveries,late_delivery_pct
AL,36,12,33.33
RJ,850,111,13.06
PB,31,4,12.90
CE,77,9,11.69
PI,47,5,10.64
SC,311,31,9.97
ES,145,14,9.66
PA,69,6,8.70
RS,367,30,8.17
PE,102,8,7.84


In [0]:
%sql
INSERT INTO olist.bronze.orders
REPLACE WHERE purchase_month = DATE '2018-01-01'
SELECT *, DATE '2018-01-01' AS purchase_month
FROM read_files(
  '/Volumes/olist/source/raw_files/olist_orders_dataset.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false,
  schemaEvolutionMode => 'none'
)
WHERE CAST(order_purchase_timestamp AS TIMESTAMP) >= TIMESTAMP '2018-01-01 00:00:00'
  AND CAST(order_purchase_timestamp AS TIMESTAMP) <  TIMESTAMP '2018-02-01 00:00:00'

num_affected_rows,num_inserted_rows
7269,7269


In [0]:
dbutils.widgets.text("purchase_month", "2018-01", "Purchase month (YYYY-MM)")

In [0]:
%sql
SELECT
  CAST(CONCAT(:purchase_month, '-01') AS DATE) AS batch_start,
  ADD_MONTHS(CAST(CONCAT(:purchase_month, '-01') AS DATE), 1) AS batch_end

batch_start,batch_end
2018-01-01,2018-02-01


In [0]:
%sql
WITH batch AS (
  SELECT CAST(CONCAT(:purchase_month, '-01') AS DATE) AS batch_start
)
INSERT INTO olist.bronze.orders
REPLACE WHERE purchase_month = CAST(CONCAT(:purchase_month, '-01') AS DATE)
SELECT source_orders.*, batch.batch_start AS purchase_month
FROM read_files(
  '/Volumes/olist/source/raw_files/olist_orders_dataset.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false,
  schemaEvolutionMode => 'none'
) AS source_orders
CROSS JOIN batch
WHERE CAST(source_orders.order_purchase_timestamp AS TIMESTAMP) >= batch.batch_start
  AND CAST(source_orders.order_purchase_timestamp AS TIMESTAMP) < ADD_MONTHS(batch.batch_start, 1)

num_affected_rows,num_inserted_rows
6728,6728
